# E2E Data Validation Pipeline

This notebook executes the complete end-to-end pipeline for testing PB3 to Parquet data conversions.

## Pipeline Steps

1. **Generate Datasets**: Create test datasets covering Protobuf3 features
2. **Infer Schema**: Generate Parquet schemas from .proto files
3. **Convert Data**: Transform .pb3 files to Parquet format
4. **Validate**: Verify data integrity after conversion
5. **Report**: Generate success/failure reports

---

In [1]:
# Setup and imports
import sys
from pathlib import Path
import pandas as pd
from datetime import datetime

# Add src to path
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(project_root / 'src'))

from generator import DatasetGenerator
from schema_inference import ProtoToParquetSchemaInference
from converter import PB3ToParquetConverter
from validator import DataValidator

print("✓ Imports successful")
print(f"Project root: {project_root}")

✓ Imports successful
Project root: /Users/r39132/Projects/learn_e2e_types_pb_parquet


## Step 1: Generate Test Datasets

Generate comprehensive test datasets covering various Protobuf3 features.

In [2]:
# Step 1: Generate datasets
print("=" * 70)
print("STEP 1: GENERATING TEST DATASETS")
print("=" * 70)

generator = DatasetGenerator(output_dir=str(project_root / "datasets"))
dataset_names = generator.generate_all_datasets()

print(f"\n✓ Generated {len(dataset_names)} datasets:")
for name in dataset_names:
    print(f"  - {name}")

print("\nDataset files created:")
for name in dataset_names:
    dataset_path = project_root / "datasets" / name
    files = list(dataset_path.glob("*"))
    print(f"\n  {name}/")
    for f in sorted(files):
        print(f"    - {f.name}")

STEP 1: GENERATING TEST DATASETS

✓ Generated 8 datasets:
  - basic_types
  - nested_messages
  - repeated_fields
  - maps
  - enums
  - oneof
  - optional_fields
  - complex_nested

Dataset files created:

  basic_types/
    - __pycache__
    - basic_types.json
    - basic_types.parquet
    - basic_types.pb3
    - basic_types.proto
    - basic_types_pb2.py

  nested_messages/
    - __pycache__
    - nested_messages.json
    - nested_messages.parquet
    - nested_messages.pb3
    - nested_messages.proto
    - nested_messages_pb2.py

  repeated_fields/
    - __pycache__
    - repeated_fields.json
    - repeated_fields.parquet
    - repeated_fields.pb3
    - repeated_fields.proto
    - repeated_fields_pb2.py

  maps/
    - __pycache__
    - maps.json
    - maps.parquet
    - maps.pb3
    - maps.proto
    - maps_pb2.py

  enums/
    - __pycache__
    - enums.json
    - enums.parquet
    - enums.pb3
    - enums.proto
    - enums_pb2.py

  oneof/
    - __pycache__
    - oneof.json
    - one

## Step 2: Infer Parquet Schemas

For each dataset, infer a Parquet schema from the .proto file.

In [3]:
# Step 2: Infer schemas
print("\n" + "=" * 70)
print("STEP 2: INFERRING PARQUET SCHEMAS")
print("=" * 70)

inferencer = ProtoToParquetSchemaInference()
schemas = {}
schema_failures = []

for dataset_name in dataset_names:
    print(f"\n📋 {dataset_name}:")
    dataset_path = project_root / "datasets" / dataset_name
    proto_file = dataset_path / f"{dataset_name}.proto"
    
    schema = inferencer.infer_schema(proto_file)
    
    if schema is None:
        print(f"  ✗ Schema inference FAILED")
        print(f"    Errors: {inferencer.get_error_summary()}")
        schema_failures.append({
            'dataset': dataset_name,
            'reason': inferencer.get_error_summary()
        })
    else:
        print(f"  ✓ Schema inferred successfully")
        print(f"    Fields: {len(schema)}")
        schemas[dataset_name] = schema
        
        # Display schema
        for field in schema:
            print(f"      - {field.name}: {field.type}")

# Report schema inference results
print("\n" + "-" * 70)
print(f"Schema Inference Summary: {len(schemas)}/{len(dataset_names)} successful")
if schema_failures:
    print("\n⚠️  Schema inference failures:")
    for failure in schema_failures:
        print(f"  - {failure['dataset']}: {failure['reason']}")
    print("\n❌ Pipeline cannot proceed with failed schemas")
else:
    print("✓ All schemas inferred successfully - proceeding to conversion")


STEP 2: INFERRING PARQUET SCHEMAS

📋 basic_types:
  ✓ Schema inferred successfully
    Fields: 9
      - int32_field: int32
      - int64_field: int64
      - uint32_field: uint32
      - uint64_field: uint64
      - float_field: float
      - double_field: double
      - bool_field: bool
      - string_field: string
      - bytes_field: binary

📋 nested_messages:
  ✓ Schema inferred successfully
    Fields: 2
      - name: string
      - address: struct<street: string, city: string, zip_code: int32>

📋 repeated_fields:
  ✓ Schema inferred successfully
    Fields: 3
      - numbers: list<item: int32>
      - tags: list<item: string>
      - items: list<item: struct<name: string, price: double>>

📋 maps:
  ✓ Schema inferred successfully
    Fields: 3
      - scores: map<string, int32>
      - metadata: map<string, string>
      - prices: map<string, struct<amount: double, currency: string>>

📋 enums:
  ✓ Schema inferred successfully
    Fields: 3
      - status: int32
      - priority:

## Step 3: Convert PB3 to Parquet

Read .pb3 files and generate Parquet data files using the inferred schemas.

In [4]:
# Step 3: Convert to Parquet
print("\n" + "=" * 70)
print("STEP 3: CONVERTING PB3 TO PARQUET")
print("=" * 70)

conversion_results = {}

# Only process datasets with successful schema inference
for dataset_name in schemas.keys():
    print(f"\n🔄 {dataset_name}:")
    dataset_path = project_root / "datasets" / dataset_name
    
    pb3_file = dataset_path / f"{dataset_name}.pb3"
    parquet_file = dataset_path / f"{dataset_name}.parquet"
    
    # Get message class name (capitalize each word)
    message_class_name = ''.join(word.capitalize() for word in dataset_name.split('_'))
    
    # Create converter with inferred schema
    converter = PB3ToParquetConverter(schemas[dataset_name])
    
    # Perform conversion
    success = converter.convert(
        pb3_file=pb3_file,
        parquet_file=parquet_file,
        proto_module_path=dataset_path,
        message_class_name=message_class_name
    )
    
    if success:
        print(f"  ✓ Conversion successful")
        print(f"    Output: {parquet_file.name}")
        
        # Show file size
        pb3_size = pb3_file.stat().st_size
        parquet_size = parquet_file.stat().st_size
        print(f"    Size: PB3={pb3_size} bytes, Parquet={parquet_size} bytes")
        
        conversion_results[dataset_name] = {
            'success': True,
            'pb3_file': pb3_file,
            'parquet_file': parquet_file,
            'pb3_size': pb3_size,
            'parquet_size': parquet_size
        }
    else:
        print(f"  ✗ Conversion FAILED")
        conversion_results[dataset_name] = {'success': False}

print("\n" + "-" * 70)
successful_conversions = sum(1 for r in conversion_results.values() if r['success'])
print(f"Conversion Summary: {successful_conversions}/{len(schemas)} successful")


STEP 3: CONVERTING PB3 TO PARQUET

🔄 basic_types:
  ✓ Conversion successful
    Output: basic_types.parquet
    Size: PB3=127 bytes, Parquet=2733 bytes

🔄 nested_messages:
  ✓ Conversion successful
    Output: nested_messages.parquet
    Size: PB3=42 bytes, Parquet=1360 bytes

🔄 repeated_fields:
  ✓ Conversion successful
    Output: repeated_fields.parquet
    Size: PB3=93 bytes, Parquet=1775 bytes

🔄 maps:
  ✓ Conversion successful
    Output: maps.parquet
    Size: PB3=131 bytes, Parquet=2761 bytes

🔄 enums:
  ✓ Conversion successful
    Output: enums.parquet
    Size: PB3=53 bytes, Parquet=1087 bytes

🔄 oneof:
  ✓ Conversion successful
    Output: oneof.parquet
    Size: PB3=32 bytes, Parquet=1166 bytes

🔄 optional_fields:
  ✓ Conversion successful
    Output: optional_fields.parquet
    Size: PB3=50 bytes, Parquet=1318 bytes

🔄 complex_nested:
  ✓ Conversion successful
    Output: complex_nested.parquet
    Size: PB3=164 bytes, Parquet=3433 bytes

---------------------------------

## Step 4: Validate Conversions

Verify that the data in PB3 and Parquet files match exactly.

In [5]:
# Step 4: Validate conversions
print("\n" + "=" * 70)
print("STEP 4: VALIDATING DATA INTEGRITY")
print("=" * 70)

validator = DataValidator()
validation_results = {}

for dataset_name, conv_result in conversion_results.items():
    if not conv_result['success']:
        print(f"\n⏭️  {dataset_name}: Skipped (conversion failed)")
        continue
    
    print(f"\n🔍 {dataset_name}:")
    
    dataset_path = project_root / "datasets" / dataset_name
    message_class_name = ''.join(word.capitalize() for word in dataset_name.split('_'))
    
    result = validator.validate(
        pb3_file=conv_result['pb3_file'],
        parquet_file=conv_result['parquet_file'],
        proto_module_path=dataset_path,
        message_class_name=message_class_name
    )
    
    validation_results[dataset_name] = result
    
    print(f"  {result}")

print("\n" + "-" * 70)
successful_validations = sum(1 for r in validation_results.values() if r.success)
print(f"Validation Summary: {successful_validations}/{len(validation_results)} passed")


STEP 4: VALIDATING DATA INTEGRITY

🔍 basic_types:
  ✓ Validation passed: 2 records match perfectly

🔍 nested_messages:
  ✓ Validation passed: 1 records match perfectly

🔍 repeated_fields:
  ✓ Validation passed: 1 records match perfectly

🔍 maps:
  ✓ Validation passed: 1 records match perfectly

🔍 enums:
  ✓ Validation passed: 2 records match perfectly

🔍 oneof:
  ✓ Validation passed: 3 records match perfectly

🔍 optional_fields:
  ✓ Validation passed: 2 records match perfectly

🔍 complex_nested:
  ✓ Validation passed: 1 records match perfectly

----------------------------------------------------------------------
Validation Summary: 8/8 passed


## Step 5: Generate Final Report

Create a comprehensive report of all pipeline results.

In [6]:
# Step 5: Generate report
print("\n" + "=" * 70)
print("STEP 5: FINAL REPORT")
print("=" * 70)

report_data = []

for dataset_name in dataset_names:
    # Determine status for each step
    schema_ok = dataset_name in schemas
    conversion_ok = dataset_name in conversion_results and conversion_results[dataset_name]['success']
    validation_ok = dataset_name in validation_results and validation_results[dataset_name].success
    
    overall_success = schema_ok and conversion_ok and validation_ok
    
    report_data.append({
        'Dataset': dataset_name,
        'Schema Inference': '✓' if schema_ok else '✗',
        'PB3→Parquet': '✓' if conversion_ok else '✗',
        'Validation': '✓' if validation_ok else '✗' if dataset_name in validation_results else '-',
        'Overall': '✓ PASS' if overall_success else '✗ FAIL',
    })

# Create DataFrame for nice display
report_df = pd.DataFrame(report_data)
print("\n")
print(report_df.to_string(index=False))

# Summary statistics
print("\n" + "=" * 70)
print("SUMMARY")
print("=" * 70)

total = len(dataset_names)
passed = sum(1 for row in report_data if row['Overall'] == '✓ PASS')
failed = total - passed

print(f"\n📊 Overall Results:")
print(f"   Total Datasets: {total}")
print(f"   Passed: {passed} ({passed/total*100:.1f}%)")
print(f"   Failed: {failed} ({failed/total*100:.1f}%)")

print(f"\n📁 Output Location: {project_root / 'datasets'}")

print(f"\n⏰ Pipeline completed at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

if failed == 0:
    print("\n🎉 All datasets passed end-to-end conversion successfully!")
else:
    print(f"\n⚠️  {failed} dataset(s) failed - review details above")


STEP 5: FINAL REPORT


        Dataset Schema Inference PB3→Parquet Validation Overall
    basic_types                ✓           ✓          ✓  ✓ PASS
nested_messages                ✓           ✓          ✓  ✓ PASS
repeated_fields                ✓           ✓          ✓  ✓ PASS
           maps                ✓           ✓          ✓  ✓ PASS
          enums                ✓           ✓          ✓  ✓ PASS
          oneof                ✓           ✓          ✓  ✓ PASS
optional_fields                ✓           ✓          ✓  ✓ PASS
 complex_nested                ✓           ✓          ✓  ✓ PASS

SUMMARY

📊 Overall Results:
   Total Datasets: 8
   Passed: 8 (100.0%)
   Failed: 0 (0.0%)

📁 Output Location: /Users/r39132/Projects/learn_e2e_types_pb_parquet/datasets

⏰ Pipeline completed at: 2026-07-13 16:13:37

🎉 All datasets passed end-to-end conversion successfully!


## Additional Analysis

View sample data from converted Parquet files.

In [7]:
# Display sample data from successfully converted datasets
print("=" * 70)
print("SAMPLE DATA FROM PARQUET FILES")
print("=" * 70)

for dataset_name in list(schemas.keys())[:3]:  # Show first 3 datasets
    if dataset_name in conversion_results and conversion_results[dataset_name]['success']:
        print(f"\n📄 {dataset_name}:")
        parquet_file = conversion_results[dataset_name]['parquet_file']
        
        df = pd.read_parquet(parquet_file)
        print(f"\n   Shape: {df.shape[0]} rows × {df.shape[1]} columns")
        print(f"\n   Columns: {', '.join(df.columns.tolist())}")
        print(f"\n   First record:")
        print(df.head(1).to_string(index=False))
        print()

SAMPLE DATA FROM PARQUET FILES

📄 basic_types:

   Shape: 2 rows × 9 columns

   Columns: int32_field, int64_field, uint32_field, uint64_field, float_field, double_field, bool_field, string_field, bytes_field

   First record:
 int32_field         int64_field  uint32_field         uint64_field  float_field  double_field  bool_field     string_field    bytes_field
          42 9223372036854775807    4294967295 18446744073709551615         3.14      2.718282        True Hello, Protobuf! b'binary data'


📄 nested_messages:

   Shape: 1 rows × 2 columns

   Columns: name, address

   First record:
    name                                                             address
John Doe {'street': '123 Main St', 'city': 'Springfield', 'zip_code': 12345}


📄 repeated_fields:

   Shape: 1 rows × 3 columns

   Columns: numbers, tags, items

   First record:
                numbers                        tags                                                                                           

---

## Next Steps

- Review the markdown documentation in `docs/` for detailed information about each dataset
- Examine failed conversions (if any) to understand Protobuf3 features that need special handling
- Extend the pipeline with additional test cases as needed